# 🕌 Bulaq 1280 AH ByT5 Arabic OCR Corrector — Google Colab Pro Trainer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/youssefbouhaik/bulaq-ocr-transformer/blob/main/Train_Bulaq_Transformer_Colab.ipynb)

Fine-tunes **`google/byt5-small`** (Byte-level T5 Transformer) on **28,468 empirical Bulaq 1280 AH lithographic OCR fallacies**.

### ⚡ Why ByT5?
Traditional subword tokenizers (like BERT or standard T5) fail on corrupted historical Arabic OCR because broken ligatures, missing dots (i'jam), and noise generate `<unk>` tokens. **ByT5 operates directly on raw UTF-8 bytes**, giving it a 0% Out-Of-Vocabulary rate and allowing it to learn character-level and ligature-level healing effortlessly.

## 1. Check GPU Acceleration
*Recommended Colab Pro Runtime: **A100 GPU** or **L4 GPU** (under Runtime -> Change runtime type)*

In [ ]:
!nvidia-smi

## 2. Install Required Packages & Clone Repo

In [ ]:
!pip install -q transformers[torch] datasets accelerate evaluate jiwer
!git clone https://github.com/youssefbouhaik/bulaq-ocr-transformer.git
%cd bulaq-ocr-transformer

## 3. Load & Inspect Bulaq Fallacies Dataset

In [ ]:
import json
from datasets import Dataset

data_path = 'data/bulaq_ocr_fallacies_dataset.jsonl'
pairs = []
with open(data_path, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            row = json.loads(line)
            src = row.get('corrupt_ocr', '').strip()
            tgt = row.get('ground_truth', '').strip()
            if src and tgt and src != tgt:
                pairs.append({'input_text': src, 'target_text': tgt})

print(f'Loaded {len(pairs):,} unique fallacy correction pairs!')

# Show sample pairs
for i in range(5):
    print(f"{i+1}. OCR: '{pairs[i]['input_text']}'  -->  Truth: '{pairs[i]['target_text']}'")

# Split 90% train, 10% validation
raw_ds = Dataset.from_list(pairs).train_test_split(test_size=0.1, seed=42)
train_ds = raw_ds['train']
val_ds = raw_ds['test']
print(f'Train: {len(train_ds):,} | Val: {len(val_ds):,}')

## 4. Initialize ByT5 Model and Tokenizer

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq

model_name = 'google/byt5-small'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print(f'Model loaded: {model_name} ({sum(p.numel() for p in model.parameters()):,} parameters)')

## 5. Byte-Level Preprocessing

In [ ]:
max_length = 128

def preprocess_function(batch):
    model_inputs = tokenizer(batch['input_text'], max_length=max_length, padding='max_length', truncation=True)
    labels = tokenizer(text_target=batch['target_text'], max_length=max_length, padding='max_length', truncation=True)
    # Replace padding token id with -100 so it is ignored by the loss
    labels['input_ids'] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels['input_ids']
    ]
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

print('Tokenizing datasets...')
tokenized_train = train_ds.map(preprocess_function, batched=True, remove_columns=['input_text', 'target_text'])
tokenized_val = val_ds.map(preprocess_function, batched=True, remove_columns=['input_text', 'target_text'])
print('Ready for training!')

## 6. Training Configuration & Execution
*With an A100 GPU and mixed precision (BF16), 3 epochs will take approximately ~10–12 minutes.*

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
import torch

use_bf16 = torch.cuda.is_bf16_supported() if (torch.cuda.is_available() and hasattr(torch.cuda, "is_bf16_supported")) else False
use_fp16 = not use_bf16 and torch.cuda.is_available()

# Universal Training Arguments (Compatible with all Colab / Transformers versions)
training_args = Seq2SeqTrainingArguments(
    output_dir="./bulaq_byt5_checkpoints",
    save_strategy="epoch",
    learning_rate=5e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=use_fp16,
    bf16=use_bf16,
    logging_steps=50,
    warmup_steps=100,
    report_to="none"
)

# Safely set evaluation strategy for both old and new Transformers versions
if hasattr(training_args, "eval_strategy"):
    training_args.eval_strategy = "epoch"
elif hasattr(training_args, "evaluation_strategy"):
    training_args.evaluation_strategy = "epoch"

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    tokenizer=tokenizer
)

# 🚀 START TRAINING
trainer.train()


## 7. Test The Model on Unseen OCR Samples

In [ ]:
def correct_text(raw_ocr):
    inputs = tokenizer(raw_ocr, return_tensors='pt', max_length=128, truncation=True).to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=128, num_beams=4, early_stopping=True)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

test_cases = [
    "بلذنى يا الماك السعيد",
    "الملا كس لسانقا ل",
    "بولدىكاذما كان",
    "صارعبدة الملبا نمع أبيهضوء المكان",
    "سعد انتوشتكاليباحالة",
    "واللهلقدضاةت ب الاأرض لا جلغيبتك"
]

print('=== INFERENCE DEMONSTRATION ===\n')
for raw in test_cases:
    healed = correct_text(raw)
    print(f'Corrupt OCR: {raw}')
    print(f'Healed Text: {healed}\n')

## 8. Save Weights (Local & Google Drive)

In [ ]:
# Save locally in session
output_model_dir = './bulaq_byt5_ocr_corrector'
model.save_pretrained(output_model_dir)
tokenizer.save_pretrained(output_model_dir)
print(f'Model saved to {output_model_dir}')

# Optional: Save directly to your Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    !mkdir -p "/content/drive/MyDrive/bulaq_byt5_ocr_corrector"
    !cp -r ./bulaq_byt5_ocr_corrector/* "/content/drive/MyDrive/bulaq_byt5_ocr_corrector/"
    print('SUCCESS: Saved model checkpoint directly to Google Drive!')
except Exception as e:
    print('Google Drive mount skipped:', e)